In [11]:
import pandas as pd
import numpy as np

print("Pandas Version:", pd.__version__)

Pandas Version: 2.2.3


 **Creating a Sample Raw Dataset**

In [12]:
#Creating a dirty sales dataset to demonstrate all cleaning steps

data = {
    'Order_ID': [101, 102, 103, 104, 105, 102, 106, 107, 108, 109],
    'Customer_Name': ['  RAhul ', ' Priya', 'Amit ', '  priya', 'Neha', ' Priya', '  RAHUL ', 'Vikash', ' Anjali ', np.nan],
    'City': ['Mumbai ', ' Delhi', 'MUMBAI', 'delhi ', np.nan, ' Delhi', 'mumbai', ' Kolkata', 'Chennai ', 'Chennai'],
    'Product': [' Laptop ', 'Mouse', ' LAPTOP', 'Mouse ', 'Keyboard', 'Mouse', 'laptop', ' Monitor ', np.nan, 'Keyboard'],
    'Quantity': [2, 1, np.nan, 1, 3, 1, 2, 5, 2, np.nan],
    'Price': [50000, 500, 50000, np.nan, 1500, 500, 50000, 8000, 1200, 1500],
    'Payment_Method': [' Cash', 'UPI ', 'cash', np.nan, ' Card', 'UPI ', 'CASH ', 'upi', 'Card ', 'card']
    }

df = pd.DataFrame(data)
# Save as raw file
df.to_csv('raw_sales.csv', index=False)
print("raw_sales.csv created successfully!")
df.head(10)

raw_sales.csv created successfully!


,Order_ID,Customer_Name,City,Product,Quantity,Price,Payment_Method
0,101,RAhul,Mumbai,Laptop,2.0,50000.0,Cash
1,102,Priya,Delhi,Mouse,1.0,500.0,UPI
2,103,Amit,MUMBAI,LAPTOP,NaN,50000.0,cash
3,104,priya,delhi,Mouse,1.0,NaN,NaN
4,105,Neha,NaN,Keyboard,3.0,1500.0,Card
5,102,Priya,Delhi,Mouse,1.0,500.0,UPI
6,106,RAHUL,mumbai,laptop,2.0,50000.0,CASH
7,107,Vikash,Kolkata,Monitor,5.0,8000.0,upi
8,108,Anjali,Chennai,NaN,2.0,1200.0,Card
9,109,NaN,Chennai,Keyboard,NaN,1500.0,card


 **Ingestion & Audit [Objective 1 & 2]**

In [13]:
#1. Load the dataset

df = pd.read_csv('raw_sales.csv')
print("--- BEFORE CLEANING ---")
print("Shape:", df.shape)
df.head()

print("\n--- Info ---")
df.info()

print("\n--- Missing Values Count ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows Count ---")
print(f"Duplicate rows: {df.duplicated().sum()}")

--- BEFORE CLEANING ---
Shape: (10, 7)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order_ID        10 non-null     int64  
 1   Customer_Name   9 non-null      object 
 2   City            9 non-null      object 
 3   Product         9 non-null      object 
 4   Quantity        8 non-null      float64
 5   Price           9 non-null      float64
 6   Payment_Method  9 non-null      object 
dtypes: float64(2), int64(1), object(4)
memory usage: 692.0+ bytes

--- Missing Values Count ---
Order_ID          0
Customer_Name     1
City              1
Product           1
Quantity          2
Price             1
Payment_Method    1
dtype: int64

--- Duplicate Rows Count ---
Duplicate rows: 1


In [14]:
#2. Remove identical repeated records
df_before = df.shape[0]
df = df.drop_duplicates()
df_after = df.shape[0]

print(f"Rows before: {df_before}")
print(f"Rows after removing duplicates: {df_after}")
print(f"Removed {df_before - df_after} duplicate rows")

Rows before: 10
Rows after removing duplicates: 9
Removed 1 duplicate rows


**Handle Missing Values - Numeric Columns**

In [15]:
#3. Impute continuous numerical columns with MEDIAN [better than mean for outliers]

for col in ['Quantity', 'Price']:
  median_val = df[col].median()
  df[col] = df[col].fillna(median_val)
  print(f"Filled missing values in '{col}' with median: {median_val}")

print("\nMissing values after numeric imputation:")
print(df.isnull().sum())

Filled missing values in 'Quantity' with median: 2.0
Filled missing values in 'Price' with median: 4750.0

Missing values after numeric imputation:
Order_ID          0
Customer_Name     1
City              1
Product           1
Quantity          0
Price             0
Payment_Method    1
dtype: int64


**Handle Missing Values - Categorical Columns**

In [16]:
#4. Impute categorical fields with MODE or a clean placeholder

for col in ['Customer_Name', 'City', 'Product', 'Payment_Method']:
  mode_val = df[col].mode()[0]
  df[col] = df[col].fillna(mode_val)
  print(f"Filled missing values in '{col}' with mode: {mode_val}")

print("\nMissing values after categorical imputation:")
print(df.isnull().sum())

Filled missing values in 'Customer_Name' with mode:   RAHUL 
Filled missing values in 'City' with mode:  Delhi
Filled missing values in 'Product' with mode: Keyboard
Filled missing values in 'Payment_Method' with mode:  Card

Missing values after categorical imputation:
Order_ID          0
Customer_Name     0
City              0
Product           0
Quantity          0
Price             0
Payment_Method    0
dtype: int64


**Text Standardization [Objective 3]**

In [17]:
#5. Standardize inconsistent string formats - stripping whitespace and lowercasing
text_columns = ['Customer_Name', 'City', 'Product', 'Payment_Method']

for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

print("Text standardization done!")
df.head(10)

Text standardization done!


,Order_ID,Customer_Name,City,Product,Quantity,Price,Payment_Method
0,101,rahul,mumbai,laptop,2.0,50000.0,cash
1,102,priya,delhi,mouse,1.0,500.0,upi
2,103,amit,mumbai,laptop,2.0,50000.0,cash
3,104,priya,delhi,mouse,1.0,4750.0,card
4,105,neha,delhi,keyboard,3.0,1500.0,card
6,106,rahul,mumbai,laptop,2.0,50000.0,cash
7,107,vikash,kolkata,monitor,5.0,8000.0,upi
8,108,anjali,chennai,keyboard,2.0,1200.0,card
9,109,rahul,chennai,keyboard,2.0,1500.0,card


**Summary Statistics & Export**

In [18]:
#6. Baseline summary statistics

print("--- AFTER CLEANING ---")
print("Final Shape:", df.shape)
print("\nSummary Statistics (describe):")
display(df.describe(include='all'))

#7. Export your final sanitized dataset

df.to_csv('cleaned_sales.csv', index=False)
print("\ncleaned_sales.csv exported successfully!")

#Verify the cleaned file

cleaned_df = pd.read_csv('cleaned_sales.csv')
cleaned_df.head()

--- AFTER CLEANING ---
Final Shape: (9, 7)

Summary Statistics (describe):


,Order_ID,Customer_Name,City,Product,Quantity,Price,Payment_Method
count,9.000000,9,9,9,9.000000,9.000000,9
unique,NaN,6,4,4,NaN,NaN,3
top,NaN,rahul,mumbai,laptop,NaN,NaN,card
freq,NaN,3,3,3,NaN,NaN,4
mean,105.000000,NaN,NaN,NaN,2.222222,18605.555556,NaN
std,2.738613,NaN,NaN,NaN,1.201850,23657.060844,NaN
min,101.000000,NaN,NaN,NaN,1.000000,500.000000,NaN
25%,103.000000,NaN,NaN,NaN,2.000000,1500.000000,NaN
50%,105.000000,NaN,NaN,NaN,2.000000,4750.000000,NaN
75%,107.000000,NaN,NaN,NaN,2.000000,50000.000000,NaN



cleaned_sales.csv exported successfully!


,Order_ID,Customer_Name,City,Product,Quantity,Price,Payment_Method
0,101,rahul,mumbai,laptop,2.0,50000.0,cash
1,102,priya,delhi,mouse,1.0,500.0,upi
2,103,amit,mumbai,laptop,2.0,50000.0,cash
3,104,priya,delhi,mouse,1.0,4750.0,card
4,105,neha,delhi,keyboard,3.0,1500.0,card


**Before/After Comparison for README**

In [19]:
print(f"Before Cleaning Shape: ({df_before}, {df.shape[1]})")
print(f"After Cleaning Shape: {df.shape}")
print(f"Total Duplicates Removed: {df_before - df_after}")
print(f"Missing Values Before: Check Cell 3 output")
print(f"Missing Values After: 0 in all columns")

Before Cleaning Shape: (10, 7)
After Cleaning Shape: (9, 7)
Total Duplicates Removed: 1
Missing Values Before: Check Cell 3 output
Missing Values After: 0 in all columns


In [20]:
#raw_sales.csv & cleaned_sales.csv files

from google.colab import files
files.download('cleaned_sales.csv')
files.download('raw_sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>